# <font color="steelblue">Severidad de cáncer de pulmón</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.


## <font color="steelblue">Objetivos</font>

Este proyecto entrena, sobre todo, una competencia que no aparece en los demás: el **pensamiento crítico sobre los datos y el planteamiento del problema**. Antes de modelar nada, tendréis que **cuestionar la tarea**:

* **No hay controles sanos.** Todos los registros son pacientes **con** cáncer; lo que cambia es el **nivel de severidad** (`Level`: Low / Medium / High). Por tanto **no se puede** construir un clasificador "cáncer / no cáncer" a partir de este dataset (no existen ejemplos negativos). La **única variable objetivo válida es `Level`**, y el problema es **multiclase ordinal de severidad**.
* **Cuidado con el código de carga:** elimina `Level`, que es precisamente el objetivo. En este guión **lo conservamos**.
* **Desconfiad de la perfección.** Es un dataset muy limpio (posiblemente **sintético/simulado**): es habitual ver *accuracy* ~100 %. Eso obliga a **interrogar el resultado** y a discutir su **generalización**.

Una vez bien planteado, aplicaréis el flujo completo: comparación de modelos, **métricas conscientes del orden**, equilibrado, optimización, combinación, interpretación y despliegue.

## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto procede de Kaggle (*The Devastator — «Cancer Patients and Air Pollution: A New Link»*) y recoge, en su versión original, **1.000 registros y 26 columnas**. Cada fila representa a un paciente y describe su perfil demográfico, su exposición a factores de riesgo ambientales y de estilo de vida, y la presencia e intensidad de una serie de síntomas respiratorios. Tras eliminar los **identificadores** (`index`, `Patient Id`) y algunos síntomas de escaso valor discriminante (`Frequent Cold`, `Dry Cough`, `Snoring`), quedan **20 predictoras** y la variable objetivo. El fichero **no contiene valores faltantes**, lo que simplifica notablemente el preprocesado.

### <font color="steelblue">Diccionario de variables</font>

**Variable objetivo**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `Level` | Categórica **ordinal** | `Low` < `Medium` < `High` | Nivel de riesgo o gravedad asignado al paciente. Es la respuesta a predecir. Su carácter **ordinal** implica que los errores entre niveles contiguos son menos graves que entre los extremos. |

**Bloque 1 — Demografía**

| Variable | Tipo | Escala | Descripción |
|---|---|---|---|
| `Age` | Numérica **continua** (entero) | años | Edad del paciente. Única predictora genuinamente continua del conjunto. |
| `Gender` | Categórica **nominal** (codificada) | 1 = hombre, 2 = mujer | Sexo del paciente. Aunque esté codificada con números, **no es ordinal**: el 2 no es «más» que el 1. |

**Bloque 2 — Riesgo ambiental, genético y hábitos de vida** *(ordinales; a mayor valor, mayor exposición o intensidad)*

| Variable | Escala | Descripción |
|---|---|---|
| `Air Pollution` | 1–8 | Grado de exposición del paciente a la contaminación atmosférica. **No es una concentración de partículas**, sino un índice atribuido a cada individuo según su entorno. |
| `Alcohol use` | 1–8 | Nivel de consumo de alcohol. |
| `Dust Allergy` | 1–8 | Intensidad de la alergia al polvo que presenta el paciente. |
| `OccuPational Hazards` | 1–8 | Grado de exposición a riesgos laborales (sustancias, ambientes o procesos nocivos en el puesto de trabajo). *(La mayúscula intercalada del nombre procede del fichero original.)* |
| `Genetic Risk` | 1–7 | Riesgo genético o predisposición hereditaria, típicamente asociada a antecedentes familiares. |
| `Balanced Diet` | 1–7 | Grado en que la alimentación del paciente es equilibrada. **Ojo con el signo:** a diferencia del resto del bloque, aquí un valor alto describe un hábito **saludable**. |
| `Obesity` | 1–7 | Grado de obesidad del paciente. |
| `Smoking` | 1–8 | Nivel de consumo de tabaco (tabaquismo **activo**). |
| `Passive Smoker` | 1–8 | Grado de exposición al humo del tabaco ajeno (tabaquismo **pasivo**). |

**Bloque 3 — Síntomas y signos clínicos** *(ordinales; a mayor valor, mayor intensidad del síntoma)*

| Variable | Escala | Descripción |
|---|---|---|
| `chronic Lung Disease` | 1–7 | Gravedad de una enfermedad pulmonar crónica preexistente (EPOC, bronquitis crónica, enfisema…). Es a la vez **antecedente** y **signo**. |
| `Chest Pain` | 1–9 | Intensidad del dolor torácico. |
| `Coughing of Blood` | 1–9 | Intensidad de la **hemoptisis** (expectoración con sangre); clínicamente, uno de los signos de alarma más específicos. |
| `Fatigue` | 1–9 | Grado de fatiga o cansancio general. |
| `Weight Loss` | 1–8 | Magnitud de la pérdida de peso involuntaria, un signo sistémico frecuente en procesos oncológicos. |
| `Shortness of Breath` | 1–9 | Intensidad de la **disnea** o dificultad respiratoria. |
| `Wheezing` | 1–8 | Intensidad de las **sibilancias** (silbido audible al respirar, propio de la obstrucción bronquial). |
| `Swallowing Difficulty` | 1–8 | Grado de **disfagia** o dificultad para tragar, que puede indicar compresión esofágica. |
| `Clubbing of Finger Nails` | 1–9 | Grado de **acropaquia** (dedos «en palillo de tambor»), signo asociado a hipoxia crónica y a diversas patologías pulmonares. |

**Columnas descartadas**

| Variable | Motivo |
|---|---|
| `index`, `Patient Id` | **Identificadores** sin valor predictivo. Incluirlos introduciría ruido y, en el caso del índice, un riesgo de fuga de información si el fichero estuviera ordenado por la respuesta. |
| `Frequent Cold`, `Dry Cough`, `Snoring` | Síntomas **inespecíficos** (escalas 1–7), comunes a multitud de afecciones banales y con escaso poder discriminante en este contexto. |

### <font color="steelblue">La variable objetivo</font>

La respuesta es **`Level`**, una variable categórica **ordinal** con tres niveles que expresan el nivel de riesgo o gravedad: `Low` < `Medium` < `High`. Su reparto es **bastante equilibrado**, lo que evita los problemas habituales de desbalanceo: la distribución de clases se sitúa en torno a 303 casos de nivel bajo, 332 de nivel medio y 365 de nivel alto.  Ese equilibrio permite emplear la exactitud como métrica orientativa sin las cautelas que exigiría un problema desequilibrado.

> **Importante sobre la codificación:** las categóricas **ya son numéricas ordinales** (a mayor valor, mayor intensidad/exposición); **no** son texto. Por eso el preprocesado es ligero y el peso del proyecto está en el **planteamiento** y la **evaluación crítica**.

### <font color="steelblue">Advertencias metodológicas (léanse antes de modelar)</font>

Este conjunto es excelente como **material docente**, pero presenta varias particularidades que conviene tener presentes y que constituyen, de hecho, buena parte del interés del proyecto:

1. **Procedencia no documentada.** La ficha de Kaggle no identifica el estudio clínico, el protocolo de recogida ni la población de origen. En consecuencia, **no debe interpretarse como una fuente de evidencia epidemiológica** sobre la

## <font color="steelblue">El conjunto de datos</font>

**1.001 registros**, 26 columnas originales (Kaggle, *The Devastator — "Cancer Patients and Air Pollution"*). Tras quitar identificadores y algún síntoma, quedan ~20 predictoras + el objetivo. **Sin valores faltantes.**

* **Demografía:** `Age` (entero), `Gender` (1 = hombre, 2 = mujer).
* **Riesgo ambiental y hábitos (ordinales, escalas 1–8/1–7):** `Air Pollution`, `Alcohol use`, `Dust Allergy`, `OccuPational Hazards`, `Genetic Risk`, `Balanced Diet`, `Obesity`, `Smoking`, `Passive Smoker`.
* **Síntomas/signos (ordinales, escalas 1–9/1–8):** `chronic Lung Disease`, `Chest Pain`, `Coughing of Blood`, `Fatigue`, `Weight Loss`, `Shortness of Breath`, `Wheezing`, `Swallowing Difficulty`, `Clubbing of Finger Nails`.
* **Objetivo `Level`** (categórico **ordinal**): `Low` < `Medium` < `High` (reparto bastante **equilibrado**).

> **Importante sobre la codificación:** las categóricas **ya son numéricas ordinales** (a mayor valor, mayor intensidad/exposición); **no** son texto. Por eso el preprocesado es ligero y el peso del proyecto está en el **planteamiento** y la **evaluación crítica**.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Cuestiona el planteamiento.** Justifica por qué la tarea es **severidad ordinal (`Level`)** y **no** "cáncer/no cáncer" (no hay controles). **Conserva `Level`.**
2. **El objetivo es ORDINAL** (`Low`<`Medium`<`High`): evalúa con métricas que penalicen más los errores graves (`Low`↔`High`): **Kappa cuadrático** y **MAE ordinal**, además de F1-macro.
3. **Partición estratificada**; el *test* solo se toca al final.
4. **Sin fuga en el preprocesado:** escalado/remuestreo dentro de un **`Pipeline`**.
5. **Equilibrado solo en *train***; aquí las clases están parejas, así que su efecto será pequeño (resultado válido).
6. **Desconfía del 100 %.** Si el rendimiento es casi perfecto, **investiga por qué** (datos demasiado limpios/sintéticos) y discute la **generalización**.
7. **Reproducibilidad y honestidad:** `random_state` fijado; reporta lo que no funcionó y los límites del estudio.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

> **Corrección importante:** el código original elimina `Level`, que es la **única variable objetivo válida**. Aquí quitamos solo identificadores y los síntomas que descarta el autor, pero **mantenemos `Level`**.

In [ ]:
# !pip -q install kagglehub imbalanced-learn gradio optuna scikit-learn shap
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Descarga y carga (equivalente a usar %cd $path y leer el CSV)
path = kagglehub.dataset_download("thedevastator/cancer-patients-and-air-pollution-a-new-link")
print("Ruta:", path, "| Archivos:", os.listdir(path))
lungcancer = pd.read_csv(os.path.join(path, "cancer patient data sets.csv"))

# Quitamos identificadores y los síntomas que el autor descarta, pero CONSERVAMOS 'Level' (el objetivo)
lungcancer = lungcancer.drop(columns=['index', 'Patient Id', 'Frequent Cold', 'Dry Cough', 'Snoring'])
print(f"Dimensiones: {lungcancer.shape[0]:,} filas × {lungcancer.shape[1]} columnas")
lungcancer.head()

# <font color="steelblue">Fase 1 — Comprensión crítica del problema y EDA</font>

**Tareas obligatorias**
1. **Crítica del planteamiento (clave).** Comprobad que **no hay clase "sin cáncer"**: ¿qué valores toma `Level`? Argumentad por qué la tarea correcta es **clasificar la severidad** (multiclase ordinal) y por qué "cáncer/no cáncer" **no es viable** con estos datos.
2. **Tipos.** Confirmad que las predictoras son **ordinales numéricas** (escalas 1–8/1–9) y que no hay faltantes.
3. **Objetivo.** Distribución de `Level` (¿equilibrado?). Mapeo ordinal `Low`=0 < `Medium`=1 < `High`=2.
4. **Relación con el objetivo.** ¿Qué factores se asocian a mayor severidad (`Smoking`, `Air Pollution`, `Genetic Risk`, `Coughing of Blood`…)? Boxplots por nivel y correlaciones. **Anotad si la separación parece "demasiado" limpia** (lo retomáis en la Fase 7).
5. **Conclusión:** 3–4 hallazgos, incluido vuestro juicio sobre la **validez** del dataset.

> **A responder:** ¿qué puede y qué **no** puede afirmar un modelo entrenado aquí? (pista: predice *severidad entre pacientes con cáncer*, **no** si una persona tiene cáncer).

# <font color="steelblue">Fase 2 — Preprocesado y partición</font>

Como las variables ya son numéricas ordinales y no hay faltantes, el preprocesado es breve; el esfuerzo está en el planteamiento y la evaluación.

**Tareas obligatorias**
1. **Objetivo ordinal:** codificad `Level` respetando el orden (`{'Low':0,'Medium':1,'High':2}`) y guardad el mapeo inverso.
2. `X`/`y`, **partición estratificada**.
3. **Escalado** de las predictoras para modelos de distancia/lineales (los árboles no lo necesitan), dentro de un **`Pipeline`**. Cuidado con `Gender` (binaria 1/2 → 0/1).
4. (Opcional) Comprobad si conviene tratar las escalas como **ordinales** tal cual o explorar otras representaciones.

# <font color="steelblue">Fase 3 — Modelos base y comparación (con métricas de orden)</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística (multinomial)**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Validación cruzada repetida** estratificada.
3. **Métricas conscientes del orden** además de las habituales: **F1-macro**, **Kappa cuadrático** (`cohen_kappa_score(..., weights='quadratic')`) y **MAE ordinal**.
4. **Tabla** comparativa y comentario. *(No os sorprendáis si casi todos rozan la perfección: eso mismo lo analizáis en la Fase 7.)*

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El reparto de `Level` es **bastante equilibrado**, así que el remuestreo probablemente aporte poco; aun así es obligatorio **medirlo** (material **11**) sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'`.
3. **Sobremuestreo:** **SMOTE** (dentro del `ImbPipeline`).

Reportad **F1-macro**, **QWK** y exactitud balanceada, y comentad si **aporta o no** (con clases parejas, lo esperable es que apenas cambie).

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y, preferiblemente, **`scoring` de orden** (el **Kappa cuadrático** como objetivo); búsqueda **sobre el `Pipeline`** (prefijo `clf__`).
3. (Recomendado) **CV anidada**.
4. Reportad mejores hiperparámetros y la mejora — que, si el problema ya es "fácil", puede ser **mínima** (otro dato para la reflexión de la Fase 7).

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual** (QWK/F1-macro). **Si ya estáis en el techo, combinar no mejorará** — decidlo con honestidad.
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, pensamiento crítico e interpretación</font>

**Tareas obligatorias**
1. **Métricas finales** en el *test*: **matriz de confusión** 3×3, **F1-macro**, **QWK**, **MAE ordinal**.
2. **Interroga el resultado (clave).** Es muy probable que obtengas ~**0.99–1.0**. No lo celebres: **investiga por qué**.
   * **Curva de aprendizaje:** ¿el modelo satura con **muy pocos** datos? Eso sugiere un problema "demasiado fácil".
   * **Separabilidad:** ¿unas pocas variables bastan para separar casi perfectamente las clases?
   * **Validez/origen:** discute si el dataset puede ser **sintético o simulado** y qué implica para la **generalización** a pacientes reales.
3. **Análisis del error ordinal:** confusiones `Low`↔`High` (las graves) y `recall` de `High`.
4. **Interpretabilidad (SHAP):** ¿mandan `Smoking`, `Air Pollution`, `Genetic Risk`, `Coughing of Blood`…? ¿Es clínicamente plausible o sospechosamente "redondo"?
5. **Discusión crítica (obligatoria y con peso):** qué **puede** afirmar el modelo (severidad **entre pacientes con cáncer**) y qué **no** (diagnosticar cáncer); límites del dataset y validez externa.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** con `joblib`.
2. **Función de predicción:** `predecir_severidad(...)` con los factores de riesgo/síntomas que devuelva el **nivel** (`Low/Medium/High`, con el mapeo inverso) y las **probabilidades**.
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) con deslizadores (escalas 1–8/1–9) para los factores; salida = nivel de severidad. En Colab da un **enlace público** (incluidlo).
4. (Opcional, nota extra) **Streamlit**/**FastAPI**.

> **Aviso (obligatorio en la interfaz):** herramienta **educativa**; estima **severidad en pacientes ya diagnosticados** y **no** diagnostica cáncer. El dataset puede tener validez limitada (posiblemente sintético).

# <font color="steelblue">Pistas y errores típicos</font>

* **No borres `Level`.** Es la única etiqueta válida; sin ella no hay objetivo. Y **no hay controles sanos**: la tarea es **severidad**, no "cáncer/no cáncer".
* **El objetivo está ordenado.** Usa **Kappa cuadrático** y **MAE ordinal**, no solo *accuracy*.
* **Sospecha del 100 %.** Apóyate en la **curva de aprendizaje** y en la separabilidad para argumentar que el problema es "demasiado fácil"/posiblemente sintético, y discute la **generalización**.
* **Alcance honesto:** el modelo estima severidad **entre pacientes con cáncer**; no detecta cáncer en población general.
* **Equilibrado:** con clases parejas, espera poca o ninguna mejora.
* **Despliegue:** guarda el **Pipeline entero** y respeta el orden/formato de las columnas.

# <font color="steelblue">Referencias</font>

* The Devastator (2022). *Cancer Patients and Air Pollution: A New Link*. Kaggle.
* *Performance of machine learning algorithms for lung cancer prediction: a comparative approach*. Scientific Reports, 2024.
* *Lung Cancer Risk Prediction with Machine Learning Models*. MDPI BDCC, 6(4), 139, 2022.
* WHO (2024). *Cancer: Key Facts*.
* Cuadernos del curso: *Boosting*, *Random Forest*, *SVM*, *Regresión logística múltiple*, *Equilibrando las muestras*.